# PRC2.1: GenAI for Image Classification

**Goals:**
This notebook is part of the second programming assignment (PRC2) for the course "Deep Learning for Visual Signal Processing I." In this notebook, you are required to undertake a series of tasks listed in the "TASKS" section. These tasks involve working with the CIFAR-10 dataset and utilize the `Pytorch` library for data processing

**Learning Objectives:**
* Engage with a hands-on tutorial on image synthesis and data augmentation for image classification
* Develop a practical understanding of stable diffusion pipeline, strategies to get synthetic data for handling data imbalance and implement them effectively.
* Evaluate and clearly articulate the impact of these strategies on your model's performance.


**Expected Outcomes:**
* Notebooks: Generate separate notebooks for each experiment conducted during this task.
* Report: Submit a concise report (no more than three pages) that adheres to the specified course format, summarizing your findings and analyses.

**Estimated Completion Time:** The tasks are designed to be completed within an estimated timeframe of 6-7 hours when using GPU acceleration.

---

Author1: Surname1, name1 (email1@estudiante.uam.es)

Author2: Surname2, name2 (email2@estudiante.uam.es)

---
###### Course: Deep Learning for Visual Signal Processing I
###### Master in [Artificial Intelligence for Image Processing and Computer Vision (IPCVai)](https://ipcv.eu/)
######  [Escuela Politécnica Superior](https://www.uam.es/EPS/Home.htm), [Universidad Autónoma de Madrid](https://www.uam.es/)


In [1]:
import torch, torchvision, diffusers

!python --version
print("Pytorch version =" + torch.__version__)
print("Torchvision version =" + torchvision.__version__)
print("Diffusers version=" + diffusers.__version__)

!pip install torch-fidelity==0.3.0

Python 3.10.12
Pytorch version =2.5.1+cu121
Torchvision version =0.20.1+cu121
Diffusers version=0.31.0


In [2]:
import torch
from diffusers import StableDiffusionPipeline

model_id = "CompVis/stable-diffusion-v1-4"
#model_id = "runwayml/stable-diffusion-v1-5"
#model_id = "stabilityai/stable-diffusion-2-1"
pipe = StableDiffusionPipeline.from_pretrained(model_id, torch_dtype=torch.float16)  

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/472 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

In [3]:
pipe = pipe.to("cuda")

# Utility functions

In [4]:
# Function to calculate accuracy of a model on a given DataLoader
def calculate_accuracy(loader, model):
    """
    Calculate the accuracy of a given model on provided data.

    Parameters:
    loader (torch.utils.data.DataLoader): The DataLoader for which to calculate the accuracy. It should provide batches of (images, labels).
    model (torch.nn.Module): The trained model to evaluate.

    Returns:
    float: The accuracy percentage of the model on the provided data.
    list: List of true labels from the dataset.
    list: List of predicted labels by the model.
    """
    import torch
    
    correct = 0  # Initialize the count of correct predictions
    total = 0    # Initialize the total count of predictions
    all_preds = []  # List to store all predictions
    all_labels = []  # List to store all true labels

    # Disable gradient computation for efficiency and safety during evaluation
    with torch.no_grad():
        # Iterate over each batch in the DataLoader
        for data in loader:
            images, labels = data  # Unpack data into images and corresponding labels
            images, labels = images.to(device), labels.to(device)  # Move data to the appropriate device (GPU or CPU)

            outputs = model(images)  # Compute the output/logits of the model for the images
            _, predicted = torch.max(outputs.data, 1)  # Get the predicted class indices (highest output values)

            total += labels.size(0)  # Increment the total count of labels processed
            correct += (predicted == labels).sum().item()  # Increment the count of correct predictions

            all_preds.extend(predicted.view(-1).cpu().numpy())  # Append current batch predictions to the list (ensure it's on CPU)
            all_labels.extend(labels.cpu().numpy())  # Append current batch labels to the list (ensure it's on CPU)

    # Calculate and return the overall accuracy, along with all labels and predictions for further analysis
    return 100 * correct / total, all_labels, all_preds
  
def compute_accuracy_per_class(train_labels, train_preds):
    """
    Computes the accuracy for each class in the dataset.

    Parameters:
    train_labels (list or torch.Tensor): A list or tensor containing the true class labels.
    train_preds (list or torch.Tensor): A list or tensor containing the predicted class labels.

    Returns:
    dict: A dictionary where each key is a class label and the corresponding value is the accuracy for that class.
    """
    import torch
    
    # Convert train_labels and train_preds to tensors if they are not already
    if isinstance(train_labels, list):
        train_labels = torch.tensor(train_labels)
    if isinstance(train_preds, list):
        train_preds = torch.tensor(train_preds)

    # Ensure predictions are in the same shape as labels
    if len(train_preds.shape) > 1:  # Check if train_preds is one-hot encoded or has probabilities
        train_preds = torch.argmax(train_preds, dim=1)  # Convert from one-hot encoding/probabilities to class labels

    # Find unique classes in the dataset
    classes = torch.unique(train_labels)
    accuracy_per_class = {}

    # Calculate accuracy for each class
    for c in classes:
        idxs = (train_labels == c)  # Get indices where the true class label is c
        class_acc = (train_preds[idxs] == train_labels[idxs]).float().mean()  # Calculate accuracy for class c
        accuracy_per_class[int(c)] = class_acc.item()  # Store the accuracy in the dictionary

    return accuracy_per_class

def print_class_accuracy(accuracy_dict, class_names, header="Accuracy per class:", samples_per_class=None):
    """
    Prints the accuracy for each class based on the dictionary provided and class names, with an optional header.
    Also prints the mean accuracy and standard deviation across all classes.

    Parameters:
    accuracy_dict (dict): A dictionary where each key is a class label index and the corresponding value is the accuracy for that class.
    class_names (list of str): A list of class names corresponding to the class label indices.
    header (str, optional): A header string to be printed before the accuracies. Default is "Accuracy per class:".
    samples_per_class (list, optional): A list of the number of samples for each class, used for detailed output.

    Returns:
    None: This function only prints output and does not return any value.
    """
    import numpy as np
    
    # Print the header
    print(header)

    # Check if the dictionary is empty
    if not accuracy_dict:
        print("No accuracy data available.")
        return

    # Calculate and store all accuracies for mean and standard deviation calculation
    accuracies = []

    # Print the accuracy for each class using the class names
    for class_index, accuracy in accuracy_dict.items():
        class_name = class_names[class_index] if class_index < len(class_names) else f"Class {class_index}"
        accuracies.append(accuracy)
        if samples_per_class is not None:
            nsamples = samples_per_class[class_index]
            ntotalsamples = np.sum(samples_per_class)
            print(f"{class_name}: {accuracy*100:.2f}% (Samples: {nsamples}/{ntotalsamples})")
        else:
            print(f"{class_name}: {accuracy*100:.2f}%")
    
    # Calculate mean and standard deviation of accuracies
    if accuracies:
        mean_accuracy = np.mean(accuracies)
        std_deviation = np.std(accuracies)
        print(f"\nMean accuracy across all classes: {mean_accuracy*100:.2f}%")
        print(f"Standard deviation of accuracies: {std_deviation*100:.2f}%")

def count_samples_dataset(dataset, full_dataset_labels):
    """Counts the number of samples per class in a dataset (handles Subset datasets)."""
    dataset_indices = dataset.indices if isinstance(dataset, torch.utils.data.Subset) else range(len(dataset))
    dataset_targets = full_dataset_labels[dataset_indices]  # Retrieve labels from full dataset

    # Count samples per class
    unique_classes, counts = torch.unique(dataset_targets, return_counts=True)
    class_counts = {int(cls): int(cnt) for cls, cnt in zip(unique_classes, counts)}

    return class_counts
    
print('Utility functions created!')

Utility functions created!


In [5]:
def create_imbalanced_data(full_dataset, samples_per_class, seed=42):
    """
    Creates an imbalanc
    ed subset of a given dataset based on specified samples per class.

    Parameters:
    full_dataset (Dataset): A PyTorch dataset object which must have a 'targets' attribute
                             that contains the class labels for the dataset items.
    samples_per_class (list of int): A list of integers where each index corresponds to a class index,
                                     and the integer value at each index represents the number of samples to
                                     include for that class in the resulting imbalanced dataset.
    seed (int, optional): A seed for the random number generator to ensure reproducibility.
                          Defaults to 42.

    Returns:
    Subset: A PyTorch Subset containing a selection of items from the full_dataset to
            form an imbalanced dataset as specified by samples_per_class.
    """

    # Import necessary modules from PyTorch and NumPy
    from torch.utils.data import Subset
    import torch
    import numpy as np
    import warnings

    # Set random seed for reproducibility
    np.random.seed(seed)

    # Obtain the indices for each class
    targets = np.array(full_dataset.targets)
    indices_per_class = {class_idx: np.where(targets == class_idx)[0] for class_idx in range(10)}

    # Determine the number of samples per class and check against available samples
    imbalanced_indices = []
    for class_idx, class_indices in indices_per_class.items():
        if samples_per_class[class_idx] > len(class_indices):
            warnings.warn(f"Requested {samples_per_class[class_idx]} samples for class {class_idx}, but only {len(class_indices)} available. Using {len(class_indices)} samples instead.")
            imbalanced_class_indices = np.random.choice(class_indices, len(class_indices), replace=False)
        else:
            imbalanced_class_indices = np.random.choice(class_indices, samples_per_class[class_idx], replace=False)
        imbalanced_indices.extend(imbalanced_class_indices)

    # Create a Subset of the full dataset using the imbalanced indices
    imbalanced_dataset = Subset(full_dataset, imbalanced_indices)
    imbalanced_dataset.targets = targets[imbalanced_indices] #create "targets" variable being employed by other functions

    return imbalanced_dataset

In [6]:
import torch

def generate_synthetic_images(prompt, num_images, 
                              model_id="bguisard/stable-diffusion-nano-2-1", #model finetuned for 128x128 images 
                              width=128, height=128,
                              seed=42, batch_size=10):
    """
    Generate synthetic images using Stable Diffusion in batches to prevent CUDA Out-of-Memory (OOM) errors.

    Args:
        prompt (str): Text prompt for generating images.
        num_images (int): Total number of images to generate.
        model_id (str): Hugging Face model ID for Stable Diffusion.
        width (int): Image width in pixels.
        height (int): Image height in pixels.
        seed (int): Random seed for reproducibility.
        batch_size (int): Number of images per batch to avoid OOM.

    Returns:
        list: List of generated PIL images.
    """
    import torch
    from diffusers import StableDiffusionPipeline
    from tqdm import tqdm  # Import tqdm for progress bar
    from transformers.utils.hub import move_cache

    move_cache() # to avoid transformers cache issues https://github.com/huggingface/transformers/issues/20428

    # Load Stable Diffusion pipeline
    pipe = StableDiffusionPipeline.from_pretrained(model_id, torch_dtype=torch.float16, safety_checker=None)  
    pipe.set_progress_bar_config(disable=True)
    pipe = pipe.to("cuda")    

    generator = torch.Generator("cuda").manual_seed(seed)  # Set random seed

    all_images = []
    
    # Generate images in batches to reduce memory usage
    with torch.no_grad():  # Disable gradient computation        
        for i in tqdm(range(0, num_images, batch_size), desc="Generating Images", unit="batch"):
            batch_prompts = [prompt] * min(batch_size, num_images - i)  # Handle last batch
            images = pipe(batch_prompts, generator=generator, height=height, width=width).images
            all_images.extend(images)  # Store generated images
            
            torch.cuda.empty_cache()  # Clear memory to prevent OOM errors

    return all_images    

# Load dataset

In [7]:
import random
from torchvision import datasets, transforms

# Define a basic transformation when loading the dataset
transform = transforms.Compose([
    transforms.ToTensor(),  # Convert images to tensor format
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))  # Normalize the images using mean and std of ImageNet       
])

# Load CIFAR-10 dataset
dataset_full = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
print("Available classes = " + str(dataset_full.classes))

100%|██████████| 170M/170M [00:45<00:00, 3.75MB/s] 


Extracting ./data/cifar-10-python.tar.gz to ./data
Available classes = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


# Create imbalance

In [8]:
# Create the imbalanced dataset for classes "cat" and "dog"
desired_samples_per_class1 = [0, 0, 0, 500, 0, 100, 0, 0, 0, 0]
dataset_imbalanced = create_imbalanced_data(dataset_full,desired_samples_per_class1)
print("Imbalanced CIFAR10 created for selected classes with " + str(len(dataset_imbalanced)) + " samples in total")

Imbalanced CIFAR10 created for selected classes with 600 samples in total


# Generating synthetic data

In [9]:
# Generate synthetic images for imbalanced class "dog"
synthetic_class_idx =5 #labels are the same
synthetic_class_name=dataset_full.classes[synthetic_class_idx]

synthetic_data = []
synthetic_labels = []
num_images = [100, 200, 400]

In [10]:
for i in range(len(num_images)):
    synthetic_data.append(generate_synthetic_images("A low-resolution image of a dog with a natural background", num_images=num_images[i]))
    synthetic_labels.append(torch.full((num_images[i],), synthetic_class_idx))
    print("Generated synthetic images = " + str(len(synthetic_data[i])))

0it [00:00, ?it/s]

model_index.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

scheduler_config.json:   0%|          | 0.00/346 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/912 [00:00<?, ?B/s]

...

Generated synthetic images = 200


0it [00:00, ?it/s]

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .
Generating Images: 100%|██████████| 40/40 [09:56<00:00, 14.91s/batch]

Generated synthetic images = 400


# Combined datasets

In [11]:
import torch 
import torchvision.transforms as transforms

# Convert images from dataset_imbalanced to tensors
real_images = torch.stack([dataset_imbalanced[i][0].clone().detach() for i in range(len(dataset_imbalanced))])
print("Real images = " + str(len(real_images)))

# Define transformation to convert PIL images to tensors
transform_synthetic = transforms.Compose([
    transforms.Resize((32, 32)),  # Resize synthetic images to 32x32
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))  # Normalize the images using mean and std of ImageNet       
])

Real images = 600


In [12]:
synthetic_images = []
combined_images = []
combined_labels = []

In [13]:
for i in range(len(num_images)):
    synthetic_images.append(torch.stack([transform_synthetic(img) if not isinstance(img, torch.Tensor) else img for img in synthetic_data[i]]))
    print("Synthetic images = " + str(len(synthetic_images[i])))
  
    # Concatenate real and synthetic data
    combined_images.append(torch.cat([synthetic_images[i], real_images]))
    combined_labels.append(torch.cat([synthetic_labels[i], torch.tensor([dataset_full.targets[i] for i in dataset_imbalanced.indices])]))
    print("Real+Synthetic images = " + str(len(combined_images[i])))

Synthetic images = 100
Real+Synthetic images = 700
Synthetic images = 200
Real+Synthetic images = 800
Synthetic images = 400
Real+Synthetic images = 1000


# Training

## Baseline

In [14]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, datasets, transforms
from torch.utils.data import DataLoader, TensorDataset, random_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42) # for reproducibility

#hyperparameters
num_workers=4
num_epochs=10
batch_size=64
lr=0.0001
weight_decay=1e-5

# Load pretrained model and change classification head
model_ft = models.resnet18(weights='IMAGENET1K_V1') # to load the model with IMAGENET pre-trained weights
model_ft.fc = nn.Linear(model_ft.fc.in_features,10) # we use an imbalanced dataset with two classes, but CIFAR10 has 10
model_ft = model_ft.to(device)
print('Pre-trained model loaded!')

# Prepare real dataset (imbalanced) for DataLoader
real_images = torch.stack([dataset_imbalanced[i][0].clone().detach() for i in range(len(dataset_imbalanced))])
real_labels = torch.tensor([dataset_full.targets[i] for i in dataset_imbalanced.indices])
real_dataset = TensorDataset(real_images,real_labels)

train_size = int(0.8 * len(real_dataset))
test_size = len(real_dataset) - train_size
train_dataset, test_dataset = random_split(real_dataset, [train_size, test_size])
train_dataset.classes = dataset_full.classes
test_dataset.classes = dataset_full.classes

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,num_workers=num_workers)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False,num_workers=num_workers)
print('Train and test partitions created!')

# Ensure dataset labels are correct and compute final classes
unique_labels = torch.unique(real_labels)
print("Unique labels in dataset:", unique_labels)
class_names = [dataset_full.classes[i] for i in unique_labels]
print("Class names:", class_names)

# Setup loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_ft.parameters(), lr=lr, weight_decay=weight_decay)
print('Training setup defined!')

# Training loop
print(f"Running training in {device} mode for {num_epochs} epochs")
for epoch in range(num_epochs):
    model_ft.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model_ft(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(train_loader.dataset)

    # evaluate epoch results
    model_ft.eval()
    train_accuracy, train_labels, train_preds = calculate_accuracy(train_loader, model_ft)
    test_accuracy, test_labels, test_preds = calculate_accuracy(test_loader, model_ft)
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}, Train Accuracy: {train_accuracy:.2f}%, Test Accuracy: {test_accuracy:.2f}%')

# get statistics for each class in the last epoch
train_accuracy_per_class = compute_accuracy_per_class(train_labels, train_preds)
test_accuracy_per_class = compute_accuracy_per_class(test_labels, test_preds)
    
# print results for each class
print_class_accuracy(accuracy_dict=train_accuracy_per_class,
                     class_names=train_dataset.classes,
                     header="TRAIN - Accuracy per class (training samples):",
                     samples_per_class=count_samples_dataset(train_dataset,real_labels))
print_class_accuracy(accuracy_dict=test_accuracy_per_class,
                     class_names=train_dataset.classes,
                     header="\nTEST - Accuracy per class (test samples):",
                     samples_per_class=count_samples_dataset(test_dataset,real_labels))

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:00<00:00, 241MB/s]


Pre-trained model loaded!
Train and test partitions created!
Unique labels in dataset: tensor([3, 5])
Class names: ['cat', 'dog']
Training setup defined!
Running training in cuda mode for 10 epochs
Epoch [1/10], Loss: 1.9850, Train Accuracy: 16.88%, Test Accuracy: 22.50%
Epoch [2/10], Loss: 1.1781, Train Accuracy: 73.54%, Test Accuracy: 52.50%
Epoch [3/10], Loss: 0.6763, Train Accuracy: 96.25%, Test Accuracy: 73.33%
Epoch [4/10], Loss: 0.3933, Train Accuracy: 99.38%, Test Accuracy: 76.67%
Epoch [5/10], Loss: 0.2166, Train Accuracy: 99.79%, Test Accuracy: 79.17%
Epoch [6/10], Loss: 0.1244, Train Accuracy: 100.00%, Test Accuracy: 80.83%
Epoch [7/10], Loss: 0.0855, Train Accuracy: 100.00%, Test Accuracy: 81.67%
Epoch [8/10], Loss: 0.0591, Train Accuracy: 100.00%, Test Accuracy: 81.67%
Epoch [9/10], Loss: 0.0405, Train Accuracy: 100.00%, Test Accuracy: 81.67%
Epoch [10/10], Loss: 0.0277, Train Accuracy: 100.00%, Test Accuracy: 82.50%
TRAIN - Accuracy per class (training samples):
cat: 100.

## Splits with synthetic data

In [15]:
from torch.utils.data import DataLoader, TensorDataset, ConcatDataset, random_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42) # for reproducibility

#hyperparameters
num_workers=4
batch_size=64

# Prepare real dataset (imbalanced) for DataLoader
real_images = torch.stack([dataset_imbalanced[i][0].clone().detach() for i in range(len(dataset_imbalanced))])
real_labels = torch.tensor([dataset_full.targets[i] for i in dataset_imbalanced.indices])
real_dataset = TensorDataset(real_images,real_labels)

train_size = int(0.8 * len(real_dataset))
test_size = len(real_dataset) - train_size
train_real_dataset, test_real_dataset = random_split(real_dataset, [train_size, test_size])
train_real_dataset.classes = dataset_full.classes
test_real_dataset.classes = dataset_full.classes

# train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,num_workers=num_workers)
# test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False,num_workers=num_workers)
print(f'Real Train ({train_size}) and test ({test_size}) partitions created!')

Real Train (480) and test (120) partitions created!


# Training with combined data

In [16]:
# Prepare synthetic dataset
for i in range(len(num_images)):
    print("\n\nNumber of synthetic images: ", num_images[i])
    print()
    synthetic_dataset = TensorDataset(synthetic_images[i], synthetic_labels[i])
    train_size = int(1*len(synthetic_images[i]))
    test_size = len(synthetic_images[i]) - train_size
    train_synthetic_dataset, test_synthetic_dataset = random_split(synthetic_dataset, [train_size, test_size])
    
    # train_loader = DataLoader(train_synthetic_dataset, batch_size=batch_size, shuffle=True,num_workers=num_workers)
    # test_loader = DataLoader(test_synthetic_dataset, batch_size=batch_size, shuffle=False,num_workers=num_workers)
    print(f'Synthetic Train ({train_size}) and test ({test_size}) partitions created!')
    
    # Prepare real+synthetic dataset
    combined_images = torch.cat([synthetic_images[i], real_images])
    combined_labels = torch.cat([synthetic_labels[i], real_labels])
    
    train_combined_dataset = ConcatDataset([train_real_dataset, train_synthetic_dataset])
    train_combined_dataset.classes = dataset_full.classes
    train_loader = DataLoader(train_combined_dataset, batch_size=batch_size, shuffle=True,num_workers=num_workers)
    test_loader = DataLoader(test_real_dataset, batch_size=batch_size, shuffle=False,num_workers=num_workers)
    print(f'Synth+Real Train ({len(train_combined_dataset)}) and test ({len(test_real_dataset)}) partitions created!')
    
    # Ensure dataset labels are correct and compute final classes
    unique_labels = torch.unique(combined_labels)
    print("Unique labels in dataset:", unique_labels)
    class_names = [dataset_full.classes[i] for i in unique_labels]
    print("Class names:", class_names)

    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torchvision import models, datasets, transforms
    
    # device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    # torch.manual_seed(42) # for reproducibility
    
    #hyperparameters
    num_epochs=10
    lr=0.0001
    weight_decay=1e-5
    
    # Load pretrained model and change classification head
    model_ft = models.resnet18(weights='IMAGENET1K_V1') # to load the model with IMAGENET pre-trained weights
    model_ft.fc = nn.Linear(model_ft.fc.in_features,10) # # we use an imbalanced dataset with two classes, but CIFAR10 has 10
    model_ft = model_ft.to(device)
    print('Pre-trained model loaded!')
    
    # Setup loss and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model_ft.parameters(), lr=lr, weight_decay=weight_decay)
    print('Training setup defined!')
    
    # Training loop
    print(f"Running training in {device} mode for {num_epochs} epochs")
    for epoch in range(num_epochs):
        model_ft.train()
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model_ft(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)
    
        epoch_loss = running_loss / len(train_loader.dataset)
    
        # evaluate epoch results
        model_ft.eval()
        train_accuracy, train_labels, train_preds = calculate_accuracy(train_loader, model_ft)
        test_accuracy, test_labels, test_preds = calculate_accuracy(test_loader, model_ft)
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}, Train Accuracy: {train_accuracy:.2f}%, Test Accuracy: {test_accuracy:.2f}%')
    
    # get statistics for each class in the last epoch
    train_accuracy_per_class = compute_accuracy_per_class(train_labels, train_preds)
    test_accuracy_per_class = compute_accuracy_per_class(test_labels, test_preds)
        
    # print results for each class
    print_class_accuracy(accuracy_dict=train_accuracy_per_class,
                         class_names=train_dataset.classes,
                         header="TRAIN - Accuracy per class (training samples):",
                         samples_per_class=count_samples_dataset(train_combined_dataset,combined_labels))
    print_class_accuracy(accuracy_dict=test_accuracy_per_class,
                         class_names=train_dataset.classes,
                         header="\nTEST - Accuracy per class (test samples):",
                         samples_per_class=count_samples_dataset(test_real_dataset,real_labels))



Number of synthetic images:  100

Synthetic Train (100) and test (0) partitions created!
Synth+Real Train (580) and test (120) partitions created!
Unique labels in dataset: tensor([3, 5])
Class names: ['cat', 'dog']
Pre-trained model loaded!
Training setup defined!
Running training in cuda mode for 10 epochs
Epoch [1/10], Loss: 1.6864, Train Accuracy: 43.62%, Test Accuracy: 26.67%
Epoch [2/10], Loss: 0.7259, Train Accuracy: 90.52%, Test Accuracy: 68.33%
Epoch [3/10], Loss: 0.3908, Train Accuracy: 95.69%, Test Accuracy: 78.33%
Epoch [4/10], Loss: 0.2365, Train Accuracy: 98.62%, Test Accuracy: 75.00%
Epoch [5/10], Loss: 0.1642, Train Accuracy: 98.97%, Test Accuracy: 75.00%
Epoch [6/10], Loss: 0.1095, Train Accuracy: 99.66%, Test Accuracy: 79.17%
Epoch [7/10], Loss: 0.0837, Train Accuracy: 99.83%, Test Accuracy: 76.67%
Epoch [8/10], Loss: 0.0763, Train Accuracy: 99.83%, Test Accuracy: 74.17%
Epoch [9/10], Loss: 0.0564, Train Accuracy: 99.83%, Test Accuracy: 77.50%
Epoch [10/10], Loss: 0

# 1.TASKS (graded with 10 points)

**PRC2.1 - Question 3: Handling Class Imbalance with Synthetic Data (4 points)**
* **Task:** Create an imbalanced dataset by selecting two CIFAR-10 classes and limiting the number of samples for one class. Generate different sets of an increasing number of synthetic images to compensate for the imbalance with a prompt based on the conclusions of the previous question. Train an image classifier using finetuning on: Real imbalanced data only; and combined real + synthetic data. Discuss any limitations or biases introduced by synthetic images.
* **Objective:** Analyze how dataset imbalance affects model accuracy. Evaluate whether synthetic images help improve classification performance for underrepresented classes.